#  `AIHUB 186.복지_분야_콜센터_상담데이터` 데이터 필터링

* 입력 폴더 위치: /home/data/data/aihub/186.복지_분야_콜센터_상담데이터/01.데이터
* 출력 폴더 위치: /home/data/expr/week2/data_prepare_aihub
* 출력 파일: asr_dataset.tsv
  * 탭으로 구분된 csv 파일 (tsv)
  * 열
      * file_id
      * audio_path
      * orgtext
      * duration_sec
      * dataset_type

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
# ============================================================
# 1. 파일 경로 설정
# ============================================================
AUDIO_ROOT = Path("/home/data/data/aihub/186.복지_분야_콜센터_상담데이터/01.데이터")
ORGTEXT_FILE = "/home/data/expr/week2/03-data_prepare_aihub/orgtext.tsv"
WAV_INFO_FILE = "/home/data/expr/week2/03-data_prepare_aihub/wav_info.tsv"

OUTPUT_FILE = Path("/home/data/expr/week2/03-data_prepare_aihub/asr_dataset.tsv")

In [ ]:
# ============================================================
# 2. 카테고리 필터 설정
# ============================================================

# 각 딕셔너리 내부는 AND 조건
# 여러 딕셔너리 사이는 OR 조건
# 예: 정신건강복지센터    > 정신건강상담         > 기타     (동작 확인 용으로 최소 데이터셋 선택함, 1.5시간)
CATEGORY_FILTERS = [
    {
        "category1": "정신건강복지센터",
        "category2": "정신건강상담",
        "category3": "기타",
    }
]

In [ ]:
# ============================================================
# 3. 문자열 정리 함수
# ============================================================

def normalize_string_series(series: pd.Series) -> pd.Series:
    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


# ============================================================
# 4. 카테고리 필터 함수
# ============================================================

def apply_category_filters(
    df: pd.DataFrame,
    category_filters: list[dict],
) -> pd.DataFrame:
    """
    한 딕셔너리 안의 조건은 AND,
    여러 딕셔너리 사이는 OR로 처리합니다.
    """

    if not category_filters:
        return df.copy()

    total_mask = pd.Series(False, index=df.index)

    for condition in category_filters:
        condition_mask = pd.Series(True, index=df.index)

        for column, expected_value in condition.items():
            if column not in df.columns:
                raise KeyError(
                    f"카테고리 열이 없습니다: {column}\n"
                    f"현재 열: {df.columns.tolist()}"
                )

            column_values = normalize_string_series(df[column])

            if isinstance(expected_value, (list, tuple, set)):
                expected_values = [
                    str(value).strip()
                    for value in expected_value
                ]

                condition_mask &= column_values.isin(expected_values)

            else:
                condition_mask &= column_values.eq(
                    str(expected_value).strip()
                )

        total_mask |= condition_mask

    return df.loc[total_mask].copy()


# ============================================================
# 5. TSV 읽기
# ============================================================

wav_df = pd.read_csv(
    WAV_INFO_FILE,
    sep="\t",
    encoding="utf-8-sig",
    dtype=str,
)

text_df = pd.read_csv(
    ORGTEXT_FILE,
    sep="\t",
    encoding="utf-8-sig",
    dtype=str,
)

# 열 이름의 공백 및 BOM 정리
wav_df.columns = (
    wav_df.columns
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

text_df.columns = (
    text_df.columns
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

print("wav_info.tsv 열:")
print(wav_df.columns.tolist())

print()

print("orgtext.tsv 열:")
print(text_df.columns.tolist())


# ============================================================
# 6. 필수 열 확인
# ============================================================

required_wav_columns = {
    "file_id",
    "relative_path",
}

required_text_columns = {
    "file_label",
    "orgtext",
}

missing_wav_columns = required_wav_columns - set(wav_df.columns)
missing_text_columns = required_text_columns - set(text_df.columns)

if missing_wav_columns:
    raise KeyError(
        f"wav_info.tsv에 필요한 열이 없습니다: "
        f"{sorted(missing_wav_columns)}\n"
        f"현재 열: {wav_df.columns.tolist()}"
    )

if missing_text_columns:
    raise KeyError(
        f"orgtext.tsv에 필요한 열이 없습니다: "
        f"{sorted(missing_text_columns)}\n"
        f"현재 열: {text_df.columns.tolist()}"
    )


# ============================================================
# 7. file id 및 전사문 정리
# ============================================================

wav_df["file_id"] = normalize_string_series(
    wav_df["file_id"]
)

wav_df["relative_path"] = normalize_string_series(
    wav_df["relative_path"]
)

text_df["file_label"] = normalize_string_series(
    text_df["file_label"]
)

text_df["orgtext"] = normalize_string_series(
    text_df["orgtext"]
)

# 식별자 또는 경로가 비어 있는 음성 데이터 제거
wav_df = wav_df[
    wav_df["file_id"].ne("")
    & wav_df["relative_path"].ne("")
].copy()

# 식별자 또는 전사문이 비어 있는 전사 데이터 제거
text_df = text_df[
    text_df["file_label"].ne("")
    & text_df["orgtext"].ne("")
].copy()

print()
print(f"유효한 음성 정보 수: {len(wav_df):,}")
print(f"유효한 전사 데이터 수: {len(text_df):,}")


# ============================================================
# 8. 카테고리 필터 적용
# ============================================================

filtered_text_df = apply_category_filters(
    text_df,
    CATEGORY_FILTERS,
)

print(
    f"카테고리 필터 후 전사 데이터 수: "
    f"{len(filtered_text_df):,}"
)


print(
    f"카테고리 필터 후 전사 데이터 수: "
    f"{len(filtered_text_df):,}"
)


# ============================================================
# 9. 필터링된 전사에 해당하는 음성 정보만 선택
# ============================================================

selected_file_ids = (
    filtered_text_df["file_label"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
)

filtered_wav_df = wav_df[
    wav_df["file_id"].isin(selected_file_ids)
].copy()

print(
    f"카테고리에 해당하는 음성 정보 수: "
    f"{len(filtered_wav_df):,}"
)


# ============================================================
# 10. 학습/평가 데이터 구분 열 추가
# ============================================================

def get_dataset_type(relative_path: str) -> str:
    """
    relative_path에 포함된 폴더명을 기준으로
    학습/평가 데이터를 구분합니다.
    """

    normalized_path = (
        str(relative_path)
        .replace("\\", "/")
        .strip("/")
    )

    path_parts = normalized_path.split("/")

    if "1.Training" in path_parts:
        return "train"

    if "2.Validation" in path_parts:
        return "validation"

    return "unknown"


filtered_wav_df["dataset_type"] = (
    filtered_wav_df["relative_path"]
    .map(get_dataset_type)
)


# ============================================================
# 11. 실제 음성 파일 전체 경로 생성
# ============================================================

def make_audio_path(relative_path) -> str:
    """
    AUDIO_ROOT와 relative_path를 결합하여
    실제 음성 파일 절대경로를 생성합니다.
    """

    if pd.isna(relative_path):
        return ""

    normalized_path = (
        str(relative_path)
        .strip()
        .replace("\\", "/")
        .lstrip("/")
    )

    return str(
        (Path(AUDIO_ROOT) / normalized_path).resolve()
    )


filtered_wav_df["audio_path"] = (
    filtered_wav_df["relative_path"]
    .map(make_audio_path)
)


# ============================================================
# 12. 실제 음성 파일 존재 여부 확인
# ============================================================

filtered_wav_df["audio_exists"] = (
    filtered_wav_df["audio_path"]
    .map(
        lambda path: bool(path)
        and Path(path).is_file()
    )
)

print()
print(
    "필터 대상 중 실제 음성 파일 존재 수: "
    f"{filtered_wav_df['audio_exists'].sum():,}"
)

print(
    "필터 대상 중 실제 음성 파일 미존재 수: "
    f"{(~filtered_wav_df['audio_exists']).sum():,}"
)

# 실제 파일이 존재하는 행만 유지
filtered_wav_df = filtered_wav_df[
    filtered_wav_df["audio_exists"]
].copy()


# ============================================================
# 13. 음성 정보와 필터링된 전사 데이터 병합
# ============================================================

result_df = filtered_wav_df.merge(
    filtered_text_df,
    left_on="file_id",
    right_on="file_label",
    how="inner",
    validate="one_to_many",
)

print()
print(
    "음성 및 전사가 모두 존재하는 행 수: "
    f"{len(result_df):,}"
)


# ============================================================
# 14. 중복 제거
# ============================================================

result_df = result_df.drop_duplicates(
    subset=[
        "file_id",
        "audio_path",
        "orgtext",
    ]
).copy()

print(
    f"중복 제거 후 행 수: "
    f"{len(result_df):,}"
)


# ============================================================
# 15. ASR 학습용 출력 열 선택
# ============================================================

required_output_columns = [
    "file_id",
    "audio_path",
    "orgtext",
    "duration_sec",
    "dataset_type",
]

missing_output_columns = (
    set(required_output_columns)
    - set(result_df.columns)
)

if missing_output_columns:
    raise KeyError(
        f"출력에 필요한 열이 없습니다: "
        f"{sorted(missing_output_columns)}\n"
        f"현재 열: {result_df.columns.tolist()}"
    )

result_df = result_df[
    required_output_columns
].copy()


# ============================================================
# 16. 열 이름 변경
# ============================================================

result_df = result_df.rename(
    columns={
        "orgtext": "transcript",
    }
)


# ============================================================
# 17. 데이터 정리
# ============================================================

result_df["file_id"] = (
    result_df["file_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

result_df["audio_path"] = (
    result_df["audio_path"]
    .fillna("")
    .astype(str)
    .str.strip()
)

result_df["transcript"] = (
    result_df["transcript"]
    .fillna("")
    .astype(str)
    .str.strip()
)

result_df["dataset_type"] = (
    result_df["dataset_type"]
    .fillna("")
    .astype(str)
    .str.strip()
)

result_df["duration_sec"] = pd.to_numeric(
    result_df["duration_sec"],
    errors="coerce",
)

# 필수 값이 없거나 잘못된 행 제거
result_df = result_df[
    result_df["file_id"].ne("")
    & result_df["audio_path"].ne("")
    & result_df["transcript"].ne("")
    & result_df["duration_sec"].notna()
    & result_df["duration_sec"].gt(0)
    & result_df["dataset_type"].isin(
        ["train", "validation"]
    )
].copy()

# 동일 음성·전사 중복 제거
result_df = result_df.drop_duplicates(
    subset=[
        "file_id",
        "audio_path",
        "transcript",
    ]
).copy()

# ============================================================
# 18. 정렬
# ============================================================

result_df = result_df.sort_values(
    by=[
        "dataset_type",
        "file_id",
    ],
    kind="stable",
).reset_index(drop=True)

# ============================================================
# 19. 결과 저장
# ============================================================

OUTPUT_FILE = Path(OUTPUT_FILE)

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

result_df.to_csv(
    OUTPUT_FILE,
    sep="\t",
    index=False,
    encoding="utf-8-sig",
)

print()
print(f"저장 완료: {OUTPUT_FILE}")
print(f"최종 데이터 수: {len(result_df):,}")

print()
print("학습/평가 데이터 수:")
print(
    result_df["dataset_type"]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("학습/평가 데이터 시간:")
print(
    result_df
    .groupby("dataset_type")["duration_sec"]
    .agg(
        file_count="count",
        total_seconds="sum",
    )
    .assign(
        total_hours=lambda x: x["total_seconds"] / 3600
    )
    .to_string()
)

print()
print("전체 음성 길이:")
print(
    f"{result_df['duration_sec'].sum() / 3600:,.2f}시간"
)

print()
print("결과 미리보기:")
display(result_df.head(10))

In [ ]:
from IPython.display import Audio, display

sample_df = result_df.sample(
    n=min(10, len(result_df)),
    random_state=42,
)

for _, r in sample_df.iterrows():
    print("FILE_ID:", r["file_id"])
    print("TYPE:", r["dataset_type"])
    print("TEXT:", r["transcript"])
    print("DURATION:", r.get("duration_sec", ""), "sec")
    print("PATH:", r["audio_path"])

    display(Audio(filename=r["audio_path"]))

    print("-" * 80)